# Public Wallex historical data audit
Run with the project's uv Python environment after collection. All cells are offline. This audits source grain (symbol, UTC candle open, interval), content hashes, observed coverage, raw defects and one illustrative chronological boundary. It makes no strategy, split-ratio, fee, eligibility or profitability decision. Exact capture URLs and UTC timestamps are in the manifest.

In [ ]:
import hashlib, json
from pathlib import Path
from datetime import UTC, datetime, timedelta
from wallex_agent.market.dataset import load_dataset
from wallex_agent.market.historical_pack import split_bars
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data/historical/MANIFEST.json').exists())
manifest = json.loads((root / 'data/historical/MANIFEST.json').read_text())
export = root / manifest['output_directory']
manifest['name'], manifest['collection_start_utc'], manifest['collection_end_utc']

In [ ]:
summary = []
for series in manifest['series']:
    assert hashlib.sha256((export / series['file']).read_bytes()).hexdigest() == series['sha256']
    quality_bytes = (export / series['quality_file']).read_bytes()
    assert hashlib.sha256(quality_bytes).hexdigest() == series['quality_sha256']
    quality = json.loads(quality_bytes)['quality']
    summary.append({key: series[key] for key in ['symbol', 'dataset', 'row_count', 'start_utc', 'end_exclusive_utc', 'actual_interval_seconds', 'missing_bars', 'gap_count', 'quality_status', 'flags']})
summary

In [ ]:
archive = root / manifest['archive_directory']
for receipt in manifest['raw_responses']:
    assert hashlib.sha256((archive / receipt['file']).read_bytes()).hexdigest() == receipt['sha256']
raw_defects = []
for series in manifest['series']:
    evidence = json.loads((export / series['quality_file']).read_text())
    if 'chunks' in evidence:
        raw_defects.append({'symbol': series['symbol'], 'dataset': series['dataset'], **{key: evidence[key] for key in ['raw_rows', 'raw_duplicate_rows', 'raw_out_of_order_pairs', 'incomplete_rows_dropped', 'cross_chunk_duplicates', 'out_of_request_rows', 'invalid_source_rows']}})
len(manifest['raw_responses']), raw_defects

In [ ]:
minute = load_dataset(export / 'datasets/1m')
boundary = datetime(2026, 9, 1, tzinfo=UTC)
for symbol, bars in minute.bars.items():
    start, stop = bars[0].ts_open, bars[-1].ts_open + timedelta(seconds=60)
    left, right = split_bars(bars, 60, [start, boundary, stop])
    assert {bar.ts_open for bar in left}.isdisjoint(bar.ts_open for bar in right)
    assert all(bar.ts_open + timedelta(seconds=60) <= boundary for bar in left)
    assert all(bar.ts_open >= boundary for bar in right)
    assert len(left) + len(right) == len(bars)
'PASS: illustrative boundary only; feature/position leakage still requires harness checks'

In [ ]:
manifest['metadata'], manifest['limitations']